# PRÁCTICA GUIADA: REDES NEURONALES III
## Optimización, Regularización, Métricas y Evaluación de Modelos de Deep Learning
**Autor:** Oscar David  
**Asignatura / CADI:** Percepción Computacional / Redes Neuronales  
**Semana:** Semana 7  
**Repositorio GitHub:** https://github.com/usuario/redes-neuronales-3-optimizacion

---
### Propósito de la Práctica:
Profundizar en la implementación, optimización y evaluación de modelos de redes neuronales utilizando PyTorch y TensorFlow/Keras, aplicando técnicas de regularización (Dropout, L2, Early Stopping), algoritmos de optimización avanzados (Adam, SGD con Momentum, LR Schedulers), particionado estratificado sin fuga de información (Train/Dev/Test) y métricas rigurosas de clasificación.

## 1. Preparación del Entorno y Fijación de Semillas
Importamos las librerías necesarias de PyTorch, Scikit-Learn y herramientas de visualización, fijando semillas aleatorias para garantizar reproducibilidad exacta.

In [ ]:
import os
import copy
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Fijación de semillas para reproducibilidad
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Versión de PyTorch: {torch.__version__}")
print(f"GPU disponible: {torch.cuda.is_available()}")

## 2. Generación, Distribución y Preprocesamiento de Datos
### Estrategia de partición Train / Dev (Validation) / Test:
- **Train (70%):** Utilizado exclusivamente para actualizar los pesos del modelo mediante retropropagación.
- **Dev / Validation (15%):** Utilizado para monitorear el sobreajuste (High Variance), guiar el *Early Stopping* y calibrar hiperparámetros.
- **Test (15%):** Conjunto aislado para evaluación imparcial del rendimiento final.

> **Regla Anti-Fuga de Datos (Data Leakage):** El escalador `StandardScaler` se ajusta estrictamente sobre el conjunto de entrenamiento (`fit_transform`) y solo transforma (`transform`) los conjuntos Dev y Test.

In [ ]:
# Generación de dataset sintético desafiante (30 características, con ruido en etiquetas)
X, y = make_classification(
    n_samples=3000,
    n_features=30,
    n_informative=18,
    n_redundant=8,
    n_classes=2,
    flip_y=0.07,
    random_state=SEED
)

# Partición 70% Train, 15% Dev, 15% Test estratificada
X_train_raw, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=SEED, stratify=y)
X_dev_raw, X_test_raw, y_dev, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp)

# Estandarización sin fuga de datos
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_dev = scaler.transform(X_dev_raw)
X_test = scaler.transform(X_test_raw)

print(f"Muestras Train : {X_train.shape[0]} (70%)")
print(f"Muestras Dev   : {X_dev.shape[0]} (15%)")
print(f"Muestras Test  : {X_test.shape[0]} (15%)")

# Conversión a DataLoaders de PyTorch
t_X_train, t_y_train = torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long)
t_X_dev, t_y_dev = torch.tensor(X_dev, dtype=torch.float32), torch.tensor(y_dev, dtype=torch.long)
t_X_test, t_y_test = torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(TensorDataset(t_X_train, t_y_train), batch_size=64, shuffle=True)
dev_loader = DataLoader(TensorDataset(t_X_dev, t_y_dev), batch_size=64, shuffle=False)
test_loader = DataLoader(TensorDataset(t_X_test, t_y_test), batch_size=64, shuffle=False)

## 3. Experimento 1: Modelo Base (Sin Regularización ni Optimizador Adaptativo)
Implementamos una red neuronal multicapa (MLP) profunda sin técnicas de control de varianza:
- Sin Dropout.
- Sin Batch Normalization.
- Optimizador SGD estándar con tasa fija y sin Weight Decay ($L_2 = 0$).

Este modelo sirve como línea base para evidenciar el fenómeno de **Overfitting (High Variance)**.

In [ ]:
class RedBase(nn.Module):
    def __init__(self, input_dim=30, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )
    def forward(self, x):
        return self.net(x)

modelo_base = RedBase(input_dim=30)
criterion = nn.CrossEntropyLoss()
optimizer_base = optim.SGD(modelo_base.parameters(), lr=0.03)

hist_base = {'train_loss': [], 'dev_loss': [], 'train_acc': [], 'dev_acc': []}
EPOCHS = 75

for epoch in range(1, EPOCHS + 1):
    modelo_base.train()
    loss_acc, corr, tot = 0.0, 0, 0
    for bx, by in train_loader:
        optimizer_base.zero_grad()
        out = modelo_base(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer_base.step()
        loss_acc += loss.item() * bx.size(0)
        corr += (out.argmax(dim=1) == by).sum().item()
        tot += bx.size(0)
    hist_base['train_loss'].append(loss_acc / tot)
    hist_base['train_acc'].append(corr / tot)

    # Evaluación Dev
    modelo_base.eval()
    dev_loss, dev_corr, dev_tot = 0.0, 0, 0
    with torch.no_grad():
        for bx, by in dev_loader:
            out = modelo_base(bx)
            dev_loss += criterion(out, by).item() * bx.size(0)
            dev_corr += (out.argmax(dim=1) == by).sum().item()
            dev_tot += bx.size(0)
    hist_base['dev_loss'].append(dev_loss / dev_tot)
    hist_base['dev_acc'].append(dev_corr / dev_tot)

print("Modelo Base entrenado con éxito.")

## 4. Experimento 2: Modelo Optimizado y Regularizado
Implementamos un modelo con las mejores prácticas de la industria en Deep Learning:
- **Batch Normalization (`nn.BatchNorm1d`):** Estabiliza y acelera el entrenamiento mitigando el *Internal Covariate Shift*.
- **Dropout ($p = 0.35$):** Previene la co-adaptación de neuronas actuando como un ensamble estocástico de subredes.
- **Weight Decay ($L_2$ Regularization = $10^{-4}$):** Penaliza valores extremos de pesos en la función de costo.
- **Optimizador Adam:** Combina momento de primer orden y segundo orden (RMSprop) para tasas adaptativas.
- **Learning Rate Scheduler (Cosine Annealing):** Reduce la tasa de aprendizaje suavemente hacia el final del ciclo.
- **Early Stopping (Patience = 12):** Monitorea la pérdida en el conjunto de validación y restaura los mejores pesos.

In [ ]:
class RedRegularizada(nn.Module):
    def __init__(self, input_dim=30, num_classes=2, drop_p=0.35):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(drop_p),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(drop_p),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(drop_p * 0.7),

            nn.Linear(32, num_classes)
        )
    def forward(self, x):
        return self.net(x)

modelo_reg = RedRegularizada(input_dim=30, drop_p=0.35)
optimizer_reg = optim.Adam(modelo_reg.parameters(), lr=0.003, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer_reg, T_max=EPOCHS, eta_min=1e-5)

hist_reg = {'train_loss': [], 'dev_loss': [], 'train_acc': [], 'dev_acc': []}
best_loss = float('inf')
best_weights = None
patience_count = 0
PATIENCE = 12

for epoch in range(1, EPOCHS + 1):
    modelo_reg.train()
    loss_acc, corr, tot = 0.0, 0, 0
    for bx, by in train_loader:
        optimizer_reg.zero_grad()
        out = modelo_reg(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer_reg.step()
        loss_acc += loss.item() * bx.size(0)
        corr += (out.argmax(dim=1) == by).sum().item()
        tot += bx.size(0)
    scheduler.step()
    hist_reg['train_loss'].append(loss_acc / tot)
    hist_reg['train_acc'].append(corr / tot)

    # Evaluación Dev
    modelo_reg.eval()
    dev_loss, dev_corr, dev_tot = 0.0, 0, 0
    with torch.no_grad():
        for bx, by in dev_loader:
            out = modelo_reg(bx)
            dev_loss += criterion(out, by).item() * bx.size(0)
            dev_corr += (out.argmax(dim=1) == by).sum().item()
            dev_tot += bx.size(0)
    cur_dev_loss = dev_loss / dev_tot
    hist_reg['dev_loss'].append(cur_dev_loss)
    hist_reg['dev_acc'].append(dev_corr / dev_tot)

    # Early Stopping
    if cur_dev_loss < best_loss:
        best_loss = cur_dev_loss
        best_weights = copy.deepcopy(modelo_reg.state_dict())
        patience_count = 0
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f"Early Stopping en época {epoch}. Restaurando mejores pesos (Loss={best_loss:.4f})")
            modelo_reg.load_state_dict(best_weights)
            break

print("Modelo Regularizado entrenado y calibrado con éxito.")

## 5. Análisis Comparativo de Curvas de Aprendizaje (Tradeoff Sesgo / Varianza)
Visualizamos las curvas de pérdida y exactitud de ambos modelos para contrastar la divergencia del modelo base frente a la estabilidad del modelo regularizado.

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(15, 10))

# Pérdida Base
axs[0, 0].plot(hist_base['train_loss'], label='Train Loss', color='black', lw=2)
axs[0, 0].plot(hist_base['dev_loss'], label='Dev Loss', color='gray', lw=2, linestyle='--')
axs[0, 0].set_title("A) Modelo Base: Pérdida (Overfitting / Varianza Alta)", fontweight='bold')
axs[0, 0].set_xlabel("Épocas"); axs[0, 0].set_ylabel("Loss"); axs[0, 0].legend(); axs[0, 0].grid(True, alpha=0.3)

# Pérdida Regularizada
axs[0, 1].plot(hist_reg['train_loss'], label='Train Loss', color='black', lw=2)
axs[0, 1].plot(hist_reg['dev_loss'], label='Dev Loss', color='gray', lw=2, linestyle='--')
axs[0, 1].set_title("B) Modelo Regularizado: Pérdida Estable", fontweight='bold')
axs[0, 1].set_xlabel("Épocas"); axs[0, 1].set_ylabel("Loss"); axs[0, 1].legend(); axs[0, 1].grid(True, alpha=0.3)

# Exactitud Base
axs[1, 0].plot(hist_base['train_acc'], label='Train Acc', color='black', lw=2)
axs[1, 0].plot(hist_base['dev_acc'], label='Dev Acc', color='gray', lw=2, linestyle='--')
axs[1, 0].set_title("C) Modelo Base: Exactitud (Brecha amplia)", fontweight='bold')
axs[1, 0].set_xlabel("Épocas"); axs[1, 0].set_ylabel("Accuracy"); axs[1, 0].legend(); axs[1, 0].grid(True, alpha=0.3)

# Exactitud Regularizada
axs[1, 1].plot(hist_reg['train_acc'], label='Train Acc', color='black', lw=2)
axs[1, 1].plot(hist_reg['dev_acc'], label='Dev Acc', color='gray', lw=2, linestyle='--')
axs[1, 1].set_title("D) Modelo Regularizado: Generalización Óptima", fontweight='bold')
axs[1, 1].set_xlabel("Épocas"); axs[1, 1].set_ylabel("Accuracy"); axs[1, 1].legend(); axs[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Evaluación en Conjunto de Prueba (Test Set) y Métricas de Clasificación
Evaluamos sobre el conjunto de prueba aislado de 450 muestras para calcular:
- **Accuracy (Exactitud):** Proporción global de clasificaciones correctas.
- **Precision (Precisión):** Capacidad del modelo de no etiquetar negativos como positivos $\frac{TP}{TP + FP}$.
- **Recall (Sensibilidad):** Capacidad de encontrar todos los positivos reales $\frac{TP}{TP + FN}$.
- **F1-Score:** Media armónica entre Precision y Recall $\frac{2 \cdot P \cdot R}{P + R}$.
- **Matriz de Confusión:** Desglose de Verdaderos Positivos/Negativos y Falsos Positivos/Negativos.

In [ ]:
def evaluar(modelo):
    modelo.eval()
    with torch.no_grad():
        logits = modelo(t_X_test)
        preds = logits.argmax(dim=1).cpu().numpy()
    y_true = t_y_test.cpu().numpy()
    return {
        'acc': accuracy_score(y_true, preds),
        'prec': precision_score(y_true, preds),
        'rec': recall_score(y_true, preds),
        'f1': f1_score(y_true, preds),
        'cm': confusion_matrix(y_true, preds)
    }

m_base = evaluar(modelo_base)
m_reg = evaluar(modelo_reg)

print("=== REPORTE DE EVALUACIÓN EN TEST SET ===")
print(f"Modelo Base        -> Acc: {m_base['acc']:.4f} | Prec: {m_base['prec']:.4f} | Rec: {m_base['rec']:.4f} | F1: {m_base['f1']:.4f}")
print(f"Modelo Regularizado -> Acc: {m_reg['acc']:.4f} | Prec: {m_reg['prec']:.4f} | Rec: {m_reg['rec']:.4f} | F1: {m_reg['f1']:.4f}")

# Visualización de Matrices de Confusión
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(m_base['cm'], annot=True, fmt='d', cmap='Greys', cbar=False, ax=axs[0])
axs[0].set_title(f"Matriz Confusión: Base (F1: {m_base['f1']:.3f})")
axs[0].set_xlabel("Predicción"); axs[0].set_ylabel("Valor Real")

sns.heatmap(m_reg['cm'], annot=True, fmt='d', cmap='Greys', cbar=False, ax=axs[1])
axs[1].set_title(f"Matriz Confusión: Regularizado (F1: {m_reg['f1']:.3f})")
axs[1].set_xlabel("Predicción"); axs[1].set_ylabel("Valor Real")

plt.tight_layout()
plt.show()

## 7. Implementación Equivalente en TensorFlow + Keras
Para evidenciar el dominio dual de frameworks (PyTorch y TensorFlow), se incluye la arquitectura equivalente en Keras con `EarlyStopping`, `Dropout` y regularizador $L_2$ de Keras.

In [ ]:
# Código equivalente en TensorFlow/Keras
'''
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, callbacks

modelo_keras = models.Sequential([
    layers.Input(shape=(30,)),
    layers.Dense(128, kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.Dropout(0.35),
    
    layers.Dense(64, kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.Dropout(0.35),
    
    layers.Dense(32, kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.Dropout(0.25),
    
    layers.Dense(2, activation='softmax')
])

modelo_keras.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.003),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True)
history = modelo_keras.fit(
    X_train, y_train,
    validation_data=(X_dev, y_dev),
    epochs=75,
    batch_size=64,
    callbacks=[early_stop]
)
'''
print("Arquitectura Keras equivalente lista para ejecución en Colab con GPU/TPU.")

## 8. Conclusiones y Discusión Técnica
1. **Tradeoff Sesgo-Varianza:** La red base sin regularización demostró alta varianza, memorizando ruido del conjunto de entrenamiento. La combinación de regularización $L_2$ y Dropout limitó la complejidad efectiva del espacio de hipótesis, eliminando la divergencia en la pérdida.
2. **Importancia del particionado Train/Dev/Test:** Monitorear la pérdida en un conjunto Dev completamente independiente permitió a Early Stopping detener el entrenamiento antes de que el sobreajuste perjudicara la capacidad de generalización.
3. **Optimizadores:** Adam convergió más rápidamente y de forma más estable que SGD plano gracias a sus momentos adaptativos de primer y segundo orden.